# ClinicalTrialEnv - Live Adaptive Trial Demo

A clinical statistician RL agent navigating a Phase II trial.

Try it live: [HF Space](https://huggingface.co/spaces/manasdutta04/clinicaltrialenv)

In [ ]:
!pip install openenv-core websockets requests matplotlib

In [ ]:
import asyncio
import json

import requests
import websockets
from IPython.display import HTML, display

WS_URL = "wss://manasdutta04-clinicaltrialenv.hf.space/ws"
HTTP_URL = "https://manasdutta04-clinicaltrialenv.hf.space"


def heuristic_action(obs):
    probs = {
        "low": obs.get("prob_low_beats_control", 0.5) if obs.get("low_active", True) else 0.0,
        "mid": obs.get("prob_mid_beats_control", 0.5) if obs.get("mid_active", True) else 0.0,
        "high": obs.get("prob_high_beats_control", 0.5) if obs.get("high_active", True) else 0.0,
    }
    total = sum(probs.values()) + 0.3
    drop = None
    for arm, ae_rate in (("low", obs.get("low_ae_rate", 0.0)), ("mid", obs.get("mid_ae_rate", 0.0)), ("high", obs.get("high_ae_rate", 0.0))):
        if ae_rate > 0.20 and obs.get(f"{arm}_active", True):
            drop = arm
            break
    interim = obs.get("interim_number", 0)
    return {
        "n_next_cohort": 25,
        "allocation_control": 0.3 / total,
        "allocation_low": probs["low"] / total,
        "allocation_mid": probs["mid"] / total,
        "allocation_high": probs["high"] / total,
        "stop_for_success": obs.get("any_arm_significant", False) and interim >= 2,
        "stop_for_futility": obs.get("futility_flag", False) and interim >= 2,
        "drop_arm": drop,
        "inclusion_criteria_strictness": 0.6,
    }


def render_table(rows):
    header = "".join(f"<th style='text-align:left;padding:8px;border-bottom:1px solid #cbd5e1'>{col}</th>" for col in ["Interim", "Patients", "p(low)", "p(mid)", "p(high)", "Resp low", "Resp mid", "Resp high"])
    body_rows = []
    for row in rows:
        body_rows.append(
            "<tr>"
            + "".join(f"<td style='padding:8px;border-bottom:1px solid #e2e8f0'>{value}</td>" for value in row)
            + "</tr>"
        )
    html = (
        "<table style='border-collapse:collapse;width:100%;font-family:monospace'>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{''.join(body_rows)}</tbody>"
        "</table>"
    )
    display(HTML(html))


async def run_demo(task_id="task_1", max_steps=30):
    history = []
    async with websockets.connect(WS_URL, ping_interval=30, ping_timeout=10) as ws:
        await ws.send(json.dumps({"type": "reset", "data": {"task_id": task_id}}))
        message = json.loads(await ws.recv())
        payload = message.get("data", message)
        observation = payload.get("observation", {})

        done = False
        step = 0
        while not done and step < max_steps:
            step += 1
            action = heuristic_action(observation)
            await ws.send(json.dumps({"type": "step", "data": action}))
            message = json.loads(await ws.recv())
            payload = message.get("data", message)
            observation = payload.get("observation", {})
            done = bool(payload.get("done", False))

            if observation.get("interim_number", 0) > 0:
                history.append({
                    "interim": observation.get("interim_number"),
                    "patients": observation.get("total_patients_enrolled"),
                    "p_low": observation.get("p_value_low"),
                    "p_mid": observation.get("p_value_mid"),
                    "p_high": observation.get("p_value_high"),
                    "resp_low": observation.get("low_response_rate"),
                    "resp_mid": observation.get("mid_response_rate"),
                    "resp_high": observation.get("high_response_rate"),
                    "reward": payload.get("reward", 0.0),
                    "done": done,
                })

    rows = [
        [
            item["interim"],
            item["patients"],
            f"{item['p_low']:.4f}",
            f"{item['p_mid']:.4f}",
            f"{item['p_high']:.4f}",
            f"{item['resp_low']:.3f}",
            f"{item['resp_mid']:.3f}",
            f"{item['resp_high']:.3f}",
        ]
        for item in history
    ]
    render_table(rows)
    return history, observation


history, final_observation = await run_demo()
print(f"Final stop reason: {final_observation.get('stop_reason')}")
print(f"Final budget remaining: {final_observation.get('budget_remaining')}")

In [ ]:
import matplotlib.pyplot as plt

interims = [item["interim"] for item in history]
plt.figure(figsize=(8, 4.5))
plt.plot(interims, [item["p_low"] for item in history], marker="o", label="Low dose")
plt.plot(interims, [item["p_mid"] for item in history], marker="o", label="Mid dose")
plt.plot(interims, [item["p_high"] for item in history], marker="o", label="High dose")
plt.axhline(0.05, color="red", linestyle="--", label="p = 0.05")
plt.ylim(1.0, 0.0)
plt.xlabel("Interim")
plt.ylabel("p-value")
plt.title("ClinicalTrialEnv: p-value trajectory on task_1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
grader_result = requests.post(f"{HTTP_URL}/grader", json={"task_id": "task_1"}, timeout=30).json()
print("Final grader score:", grader_result.get("score"))
print("Trial outcome:", grader_result.get("trial_outcome"))
print("Breakdown:")
for key, value in grader_result.get("breakdown", {}).items():
    print(f"  - {key}: {value}")

## Interpretation - what the numbers mean

- Falling p-values indicate stronger evidence that a treatment arm beats control.
- Posterior probabilities help the heuristic concentrate patients on promising arms before classical significance is reached.
- If the run ends in `success`, the agent found convincing evidence efficiently.
- If the run ends in `futility`, that can still be the right decision when the budget is too tight to justify further enrollment.
- Safety and efficiency both matter: the final grader score rewards good statistical outcomes without wasting patients.